# C3 Step 8 — E008: fine-tune the segmentation model (first REAL training run)

**What this is:** the first experiment that changes model weights — fine-tuning
the segmentation network on our scripted audio (the Indonesian-paper recipe).

**Honest expectation, stated before running (logbook D8 spirit):** our own
data says speech *detection* is near-solved (miss 1.4 %, FA 1.5 %), so the
headline DER may barely move. What this run really buys is (1) the training
machinery working end-to-end before the 15 h corpus arrives, and (2) possibly
better *overlap* handling, which the segmentation stage owns.

**Split:** train = 21 scripted clips, dev = R0023–R0026 (4 clips, used for
early stopping), TEST = the 2 IRD clips, frozen, scored once at the end.
Speakers are shared between train and dev (unavoidable — logbook D10);
TEST remains clean.

**Warning:** training code is the most version-sensitive part of pyannote.
If a cell errors, paste the FULL error into the chat — several cells have
fallbacks but do not improvise beyond them.

Setup: T4 GPU, `HF_TOKEN` secret, terms accepted for
`pyannote/segmentation-3.0` AND `pyannote/wespeaker-voxceleb-resnet34-LM`.
Budget: ~1.5–2 h (training ~40–60 min + scoring).


In [ ]:
# Cell 1 — GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# Cell 2 — install, then Runtime > Restart session, continue from Cell 3
!pip install -q pyannote.audio pyannote.metrics
print('NOW: Runtime > Restart session, then continue from Cell 3.')


In [ ]:
# Cell 3 — upload E005_upload.tgz, build the pyannote database (train/dev split)
import os, glob
if not os.path.exists('/content/E005_upload.tgz'):
    raise SystemExit('Upload E005_upload.tgz first.')
!cd /content && tar xf E005_upload.tgz

TEST = ['IRD_Harshana_Silva', 'IRD_Recording_2_IIT']       # FROZEN
DEV  = ['R0023', 'R0024', 'R0025', 'R0026']                # early stopping
truths = sorted(os.path.basename(p)[:-5] for p in glob.glob('/content/rttm_truth/*.rttm'))
TRAIN = [c for c in truths if c not in TEST and c not in DEV]
print(f'train {len(TRAIN)} / dev {len(DEV)} / test {len(TEST)} (frozen)')

os.makedirs('/content/db/lists', exist_ok=True)
for name, clips in (('train', TRAIN), ('dev', DEV)):
    with open(f'/content/db/lists/{name}.lst', 'w') as fh:
        fh.write('\n'.join(clips) + '\n')
    with open(f'/content/db/lists/{name}.rttm', 'w') as fh:
        for c in clips: fh.write(open(f'/content/rttm_truth/{c}.rttm').read())
    with open(f'/content/db/lists/{name}.uem', 'w') as fh:
        for c in clips: fh.write(open(f'/content/rttm_truth/{c}.uem').read())

with open('/content/db/database.yml', 'w') as fh:
    fh.write('''Databases:
  C3: /content/processed/{uri}.wav
Protocols:
  C3:
    SpeakerDiarization:
      pilot:
        train:
          uri: /content/db/lists/train.lst
          annotation: /content/db/lists/train.rttm
          annotated: /content/db/lists/train.uem
        development:
          uri: /content/db/lists/dev.lst
          annotation: /content/db/lists/dev.rttm
          annotated: /content/db/lists/dev.uem
''')
print('database.yml written')


In [ ]:
# Cell 4 — load protocol + pretrained segmentation model
import os
os.environ['PYANNOTE_METRICS_ENABLED'] = '0'
os.environ['PYANNOTE_DATABASE_CONFIG'] = '/content/db/database.yml'
from google.colab import userdata
from pyannote.database import registry, FileFinder
registry.load_database('/content/db/database.yml')
protocol = registry.get_protocol('C3.SpeakerDiarization.pilot',
                                 preprocessors={'audio': FileFinder(registry=registry)})

from pyannote.audio import Model
seg_model = Model.from_pretrained('pyannote/segmentation-3.0',
                                  use_auth_token=userdata.get('HF_TOKEN'))
print(seg_model.specifications)


In [ ]:
# Cell 5 — fine-tune (the slow cell: ~40-60 min)
import torch
try:
    import lightning.pytorch as pl                     # what pyannote 4.x uses
    from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
except ImportError:
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pyannote.audio.tasks import SpeakerDiarization as SegTask

# read the window length BEFORE attaching a task (a stale task makes this raise)
try:
    DUR = float(seg_model.specifications.duration)
except Exception:
    DUR = 10.0            # segmentation-3.0's training window; safe fallback
print('chunk duration:', DUR)

task = SegTask(protocol, duration=DUR,
               max_speakers_per_chunk=3, max_speakers_per_frame=2,
               batch_size=32, num_workers=2)
seg_model.task = task

try:
    monitor, mode = task.val_monitor
except Exception:
    monitor, mode = 'loss/val', 'min'
print('monitoring:', monitor, mode)

ckpt = ModelCheckpoint(dirpath='/content/ckpt', filename='best',
                       monitor=monitor, mode=mode, save_top_k=1)
stop = EarlyStopping(monitor=monitor, mode=mode, patience=10)
trainer = pl.Trainer(accelerator='gpu', devices=1, max_epochs=40,
                     callbacks=[ckpt, stop], gradient_clip_val=0.5)
trainer.fit(seg_model)
print('best checkpoint:', ckpt.best_model_path, 'val loss:', float(ckpt.best_model_score))


In [ ]:
# Cell 6 — two identical pipelines: pretrained vs fine-tuned segmentation
# Same embedding, same clustering params -> the ONLY difference is the fine-tune.
# Parameter names moved between pyannote versions, so ask the pipeline instead
# of hard-coding: default_parameters() first, then known candidate shapes.
import torch, inspect
from google.colab import userdata
from pyannote.audio import Model
from pyannote.audio.pipelines import SpeakerDiarization as SDPipeline

EMB = 'pyannote/wespeaker-voxceleb-resnet34-LM'
tok = userdata.get('HF_TOKEN')
_params = inspect.signature(SDPipeline.__init__).parameters
TOKKW = {'token': tok} if 'token' in _params else (
        {'use_auth_token': tok} if 'use_auth_token' in _params else {})

CANDIDATES = [
    {'segmentation': {'min_duration_off': 0.0},
     'clustering': {'threshold': 0.6, 'Fa': 0.07, 'Fb': 0.8}},            # 4.x style
    {'segmentation': {'min_duration_off': 0.0},
     'clustering': {'method': 'centroid', 'min_cluster_size': 12,
                    'threshold': 0.7045654963945799}},                     # 3.1 style
]

probe = SDPipeline(segmentation='pyannote/segmentation-3.0', embedding=EMB, **TOKKW)
print('search space this version expects:')
try:
    print(probe.parameters())
except Exception as e:
    print(' (could not read:', repr(e), ')')

PARAMS = None
try:
    PARAMS = probe.default_parameters()
    print('\nusing pipeline.default_parameters():', PARAMS)
except Exception:
    for cand in CANDIDATES:
        try:
            probe.instantiate(cand); PARAMS = cand
            print('\nusing candidate:', cand); break
        except ValueError as e:
            print(' rejected:', e)
if PARAMS is None:
    raise SystemExit('None of the candidate parameter shapes fit. Paste the '
                     'search space printed above into the chat.')

finetuned = Model.from_pretrained('/content/ckpt/best.ckpt')
pipes = {}
for name, seg in (('pretrained', 'pyannote/segmentation-3.0'), ('finetuned', finetuned)):
    p = SDPipeline(segmentation=seg, embedding=EMB, **TOKKW)
    p.instantiate(PARAMS)                      # identical params for both — the A/B
    p.to(torch.device('cuda'))
    pipes[name] = p
print('\nboth pipelines ready with identical parameters')


In [ ]:
# Cell 7 — score both on DEV (sanity) then ONCE on frozen TEST
import os
from pyannote.core import Annotation, Segment, Timeline
from pyannote.metrics.diarization import DiarizationErrorRate

def load_rttm(path, uri):
    ann = Annotation(uri=uri)
    for i, line in enumerate(open(path, encoding='utf-8')):
        p = line.split()
        if len(p) >= 8 and p[0] == 'SPEAKER' and float(p[4]) > 0:
            ann[Segment(float(p[3]), float(p[3]) + float(p[4])), f't{i}'] = p[7]
    return ann
def load_uem(path, uri):
    tl = Timeline(uri=uri)
    for line in open(path, encoding='utf-8'):
        p = line.split()
        if len(p) >= 4: tl.add(Segment(float(p[2]), float(p[3])))
    return tl
def score(hyps, clips, collar=0.0):
    pooled = dict.fromkeys(('confusion','missed detection','false alarm','total'), 0.0)
    per = {}
    for c in clips:
        ref = load_rttm(f'/content/rttm_truth/{c}.rttm', c)
        uem = load_uem(f'/content/rttm_truth/{c}.uem', c)
        comp = DiarizationErrorRate(collar=collar, skip_overlap=False)(
            ref, hyps[c], uem=uem, detailed=True)
        per[c] = sum(comp[k] for k in ('confusion','missed detection','false alarm'))/comp['total']*100
        for k in pooled: pooled[k] += comp[k]
    der = sum(pooled[k] for k in ('confusion','missed detection','false alarm'))/pooled['total']*100
    miss = pooled['missed detection']/pooled['total']*100
    fa   = pooled['false alarm']/pooled['total']*100
    conf = pooled['confusion']/pooled['total']*100
    return der, miss, fa, conf, per

import csv
with open('/content/e008_results.csv', 'w', newline='') as fh:
    w = csv.writer(fh); w.writerow(['pipeline','set','collar','DER','miss','fa','conf','per_clip'])
    for setname, clips in (('DEV', ['R0023','R0024','R0025','R0026']),
                           ('TEST', ['IRD_Harshana_Silva','IRD_Recording_2_IIT'])):
        for name, p in pipes.items():
            hyps = {}
            os.makedirs(f'/content/rttm_e008_{name}', exist_ok=True)
            for c in clips:
                out = p(f'/content/processed/{c}.wav')
                ann = out.speaker_diarization if hasattr(out, 'speaker_diarization') else out
                hyps[c] = ann
                with open(f'/content/rttm_e008_{name}/{c}.rttm', 'w') as f2:
                    ann.write_rttm(f2)
            for collar in (0.0, 0.25):
                der, miss, fa, conf, per = score(hyps, clips, collar)
                w.writerow([name, setname, collar, f'{der:.4f}', f'{miss:.4f}',
                            f'{fa:.4f}', f'{conf:.4f}', per])
                print(f'{setname} {name:>10} collar {collar}: DER {der:.2f} '
                      f'(miss {miss:.2f} fa {fa:.2f} conf {conf:.2f})')
print('\nREADING: compare finetuned vs pretrained on TEST — same embedding, '
      'same params, only the segmentation weights differ. Also watch whether '
      'miss/fa (the segmentation terms) moved even if DER did not.')


In [ ]:
# Cell 8 — download
!cd /content && zip -qr e008_results.zip e008_results.csv rttm_e008_pretrained rttm_e008_finetuned ckpt/best.ckpt
from google.colab import files
files.download('/content/e008_results.zip')
# Send the zip to the chat; logbook entry gets written from it.
# The .ckpt is your first trained model — keep it.
